### The purpose of this is to take people involved in movies and turn them into scores

The key idea is by looking at the median and average revenue of the movies people are in you can turn that into a score for them. There are two types of people those that are actors and those involved in production. Then the data is pivoted such that instead of multiple copies of the movie, there are just scores for production and actors in them, representing their starpower.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("movie-data/combined_df.csv")


In [3]:
# This performs a very simple pivot of the data.

input_data_df = df.copy()
input_data_df['nameAndID'] = input_data_df['primaryName'] + " (" + input_data_df['nconst'].astype(str) + ")"

# Now transfer the duplicate data into distinct columns. So director should be a column and actors is a column, etc.
basic_pivot_df = pd.DataFrame()
# basic_pivot_df = input_data_df.pivot_table(
#     index=['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
#        'revenue', 'runtime', 'adult', 'budget', 'imdb_id', 'original_language',
#        'original_title', 'overview', 'popularity', 'tagline', 'genres',
#        'production_companies', 'production_countries', 'spoken_languages',
#        'keywords'], 
#     columns="category",
#     values="nameAndID", 
#     aggfunc=lambda x: ", ".join(x)).reset_index()

print(basic_pivot_df.head(5))

Empty DataFrame
Columns: []
Index: []


In [4]:
basic_pivot_df.to_csv('movie-data/basic_pivot_df.csv', index=False)

In [5]:
# Movies contains all rows where the revenue is greater than 0.
movies_df = df[df["revenue"] > 0]
movies_df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,tagline,genres,production_companies,production_countries,spoken_languages,keywords,tconst,nconst,category,primaryName
0,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0000138,actor,Leonardo DiCaprio
1,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0330687,actor,Joseph Gordon-Levitt
2,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0680983,actor,Elliot Page
3,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0913822,actor,Ken Watanabe
4,27205,Inception,8.364,34495,Released,7/15/2010,825532764,148,False,160000000,...,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",tt1375666,nm0362766,actor,Tom Hardy


In [6]:
# A dataframe of rows only containing cast.
actor_roles = ['actor', 'actress', "self"]
actors_df = movies_df[movies_df['category'].isin(actor_roles)].copy()
# print(actors_df.head(20))

# A dataframe of rows only containing crew.
production_roles = ['director', 'producer', 'writer']
production_df = movies_df[movies_df['category'].isin(production_roles)].copy()
# print(production_df.head(20))

print("Actors/Actresses:", actors_df.shape)
print("Production:", production_df.shape)

Actors/Actresses: (164265, 25)
Production: (85897, 25)


In [7]:
# Compute profit per movie and put it in a new column.
actors_df['profit'] = actors_df['revenue'] - actors_df['budget']

# Aggregate profit data per actor
# Creates new dataframe where each row is an actor and the columns are 
# average profit, median profit, and number of titles they did.
agg_actors = actors_df.groupby(['nconst', 'primaryName']).agg(
    avg_profit=('profit', 'mean'),
    med_profit=('profit', 'median'),
    num_titles=('title', 'nunique')
).reset_index()

# The average of each actor's average profit and median profit in a column called Composite.
agg_actors['composite'] = (agg_actors['avg_profit'] + agg_actors['med_profit']) / 2

# Punish low number of titles: scale weight if num_titles < 4; otherwise weight=1
# Creates a weight column which is equal to 1 if an actor has been in 4 or more movies
# and is num_movies / 4 if they have been in less than 4.
agg_actors['weight'] = agg_actors['num_titles'].apply(lambda x: x / 4 if x < 4 else 1)

# Multiplies the composite by the weight.
agg_actors['final_composite'] = agg_actors['composite'] * agg_actors['weight']

# some statistics on the actor's scores.
mean_comp = agg_actors['final_composite'].mean()
std_dev_comp = agg_actors['final_composite'].std()
agg_actors['z_score'] = (agg_actors['final_composite'] - mean_comp) / std_dev_comp

print(mean_comp)
print(std_dev_comp)

# Transform z-score to final score; allow negative values for flops
agg_actors['final_score'] = 100 * agg_actors['z_score']

pd.set_option("display.float_format", "{:.2f}".format)

agg_actors.sort_values('final_score', ascending=False)[
    ['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 
     'weight', 'final_composite', 'z_score', 'final_score']
].head(20)


7084112.433064383
25268956.362306334


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_composite,z_score,final_score
48650,nm1569276,Chadwick Boseman,11,838737521.13,905046416.00,871891968.57,1.00,871891968.57,34.22,3422.41
51790,nm1853544,Pierre Coffin,4,773256919.00,903090397.50,838173658.25,1.00,838173658.25,32.89,3288.97
317,nm0000355,Anthony Daniels,9,640437861.67,737000000.00,688718930.83,1.00,688718930.83,26.98,2697.52
37629,nm1019674,Sala Baker,4,528959764.17,778368364.00,653664064.08,1.00,653664064.08,25.59,2558.79
26101,nm0641063,Dean O'Gorman,4,546592665.75,707209894.00,626901279.88,1.00,626901279.88,24.53,2452.88
60867,nm3269138,Dana Gaier,3,770331315.00,894761885.00,832546600.00,0.75,624409950.00,24.43,2443.02
45672,nm1388927,Miranda Cosgrove,3,770331315.00,894761885.00,832546600.00,0.75,624409950.00,24.43,2443.02
59909,nm3094377,Willow Shields,4,619547860.50,624875717.50,622211789.00,1.00,622211789.00,24.34,2434.32
36107,nm0942247,Bonnie Wright,4,533989119.25,694132532.50,614060825.88,1.00,614060825.88,24.02,2402.06
63904,nm3918035,Zendaya,6,686488344.83,527005800.50,606747072.67,1.00,606747072.67,23.73,2373.12


In [8]:
print("Aggregated Actors/Actresses:")
agg_actors.sort_values('final_score', ascending=False)[['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 'weight', 'final_score']].head(15)

Aggregated Actors/Actresses:


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_score
48650,nm1569276,Chadwick Boseman,11,838737521.13,905046416.00,871891968.57,1.00,3422.41
51790,nm1853544,Pierre Coffin,4,773256919.00,903090397.50,838173658.25,1.00,3288.97
317,nm0000355,Anthony Daniels,9,640437861.67,737000000.00,688718930.83,1.00,2697.52
37629,nm1019674,Sala Baker,4,528959764.17,778368364.00,653664064.08,1.00,2558.79
26101,nm0641063,Dean O'Gorman,4,546592665.75,707209894.00,626901279.88,1.00,2452.88
60867,nm3269138,Dana Gaier,3,770331315.00,894761885.00,832546600.00,0.75,2443.02
45672,nm1388927,Miranda Cosgrove,3,770331315.00,894761885.00,832546600.00,0.75,2443.02
59909,nm3094377,Willow Shields,4,619547860.50,624875717.50,622211789.00,1.00,2434.32
36107,nm0942247,Bonnie Wright,4,533989119.25,694132532.50,614060825.88,1.00,2402.06
63904,nm3918035,Zendaya,6,686488344.83,527005800.50,606747072.67,1.00,2373.12


In [9]:
print(agg_actors[agg_actors["primaryName"] == "Brad Pitt"])

       nconst primaryName   avg_profit  med_profit  num_titles    composite  \
73  nm0000093   Brad Pitt 115512306.36 90845033.00          44 103178669.68   

    weight  final_composite  z_score  final_score  
73    1.00     103178669.68     3.80       380.29  


In [10]:
# Compute profit per movie and put it in a new column.
production_df['profit'] = production_df['revenue'] - production_df['budget']

# Aggregate profit data per crew member (production person)
# Creates new dataframe where each row is an actor and the columns are 
# average profit, median profit, and number of titles they did.
agg_production = production_df.groupby(['nconst', 'primaryName']).agg(
    avg_profit=('profit', 'mean'),
    med_profit=('profit', 'median'),
    num_titles=('title', 'nunique')
).reset_index()

# The average of each crew member's average profit and median profit in a column called Composite.
agg_production['composite'] = (agg_production['avg_profit'] + agg_production['med_profit']) / 2

# Apply a weight that scales if num_titles < 4; otherwise, weight = 1
# Creates a weight column which is equal to 1 if an crew member has been in 4 or more movies
# and is num_movies / 4 if they have been in less than 4.
agg_production['weight'] = agg_production['num_titles'].apply(lambda x: x / 4 if x < 4 else 1)

# Multiplies the composite by the weight.
agg_production['final_composite'] = agg_production['composite'] * agg_production['weight']

# some statistics on the crew's scores.
mean_comp = agg_production['final_composite'].mean()
std_comp = agg_production['final_composite'].std()
agg_production['z_score'] = (agg_production['final_composite'] - mean_comp) / std_comp

# Transform z-score to final score; allow negative values for flops
agg_production['final_score'] = 100 * agg_production['z_score']

agg_production.sort_values('final_score', ascending=False)[
    ['nconst', 'primaryName', 'num_titles', 'avg_profit', 'med_profit', 'composite', 
     'weight', 'final_composite', 'z_score', 'final_score']
].head()


,nconst,primaryName,num_titles,avg_profit,med_profit,composite,weight,final_composite,z_score,final_score
9501,nm0484457,Jon Landau,6,1138348493.50,1047615412.00,1092981952.75,1.00,1092981952.75,27.60,2759.74
23243,nm1601644,Jennifer Lee,4,864065106.17,1124219009.00,994142057.58,1.00,994142057.58,25.08,2507.91
3010,nm0118333,Chris Buck,4,756396104.14,1124219009.00,940307556.57,1.00,940307556.57,23.71,2370.75
24507,nm1853544,Pierre Coffin,4,848431226.75,923157235.00,885794230.88,1.00,885794230.88,22.32,2231.86
20536,nm1273099,Erik Sommers,5,862747749.40,905339117.00,884043433.20,1.00,884043433.20,22.27,2227.40


### Ok now here you is where you add the columns you just calculated

In [11]:
# This adds the new actor final scores into the movies dataframe.
# So each row is a single movie and a single actor with their score. (hasn't been pivoted)
movies_with_actor_scores_df = pd.merge(movies_df, agg_actors[['nconst', 'final_score']],
                                       on='nconst', how='left', suffixes=('', '_actor'))

# Check for missing values in the actor final_score column
missing_actor_final = movies_with_actor_scores_df['final_score'].isna().sum()
print("Missing values in actor final_score:", missing_actor_final)

# This compiles statistics on each movie. So for each movie it calculates
# the average of all the actors' scores, their median, and standard deviation.
actor_stats_df = movies_with_actor_scores_df[movies_with_actor_scores_df['category'].isin(actor_roles)] \
    .groupby('tconst')['final_score'] \
    .agg(actor_avg='mean', actor_med='median', actor_dev='std') \
    .reset_index()

# If only one actor is present, std can be NaN; fill these with 0.
actor_stats_df['actor_dev'] = actor_stats_df['actor_dev'].fillna(0)

# Repeats the process above for the crew.
movies_with_prod_scores_df = pd.merge(movies_df, agg_production[['nconst', 'final_score']],
                                       on='nconst', how='left', suffixes=('', '_prod'))

# Check for missing values in the production final_score column
missing_prod_final = movies_with_prod_scores_df['final_score'].isna().sum()
print("Missing values in production final_score:", missing_prod_final)

# Repeats the process above for the crew.
prod_stats_df = movies_with_prod_scores_df[movies_with_prod_scores_df['category'].isin(production_roles)] \
    .groupby('tconst')['final_score'] \
    .agg(production_avg='mean', production_med='median', production_dev='std') \
    .reset_index()

# Fill NaN in production_dev with 0 (e.g., if only one crew member's score is present)
prod_stats_df['production_dev'] = prod_stats_df['production_dev'].fillna(0)

# Filters out everything but director, producer, and writer.
prod_only_df = movies_with_prod_scores_df[movies_with_prod_scores_df['category'].isin(production_roles)]

# Optionally, if you want to impute missing final_score in prod_only_df with the median score:
# median_prod_score = prod_only_df['final_score'].median()
# prod_only_df['final_score'] = prod_only_df['final_score'].fillna(median_prod_score)

# Check missing values in the prod_only final_score column
missing_prod_only = prod_only_df['final_score'].isna().sum()
print("Missing values in prod_only final_score:", missing_prod_only)

# Display a sample of the production-only dataframe with final scores.
prod_only_df[["title", "category", "primaryName", "final_score"]].head(10)


Missing values in actor final_score: 157517
Missing values in production final_score: 226851
Missing values in prod_only final_score: 0


,title,category,primaryName,final_score
10,Inception,director,Christopher Nolan,1034.78
11,Inception,writer,Christopher Nolan,1034.78
12,Inception,producer,Christopher Nolan,1034.78
13,Inception,producer,Emma Thomas,1113.91
30,Interstellar,director,Christopher Nolan,1034.78
31,Interstellar,writer,Jonathan Nolan,836.51
32,Interstellar,writer,Christopher Nolan,1034.78
33,Interstellar,producer,Christopher Nolan,1034.78
34,Interstellar,producer,Lynda Obst,118.17
35,Interstellar,producer,Emma Thomas,1113.91


In [12]:
# Removes all the duplicates so only one row per movie remains.
movies_unique = movies_df.drop_duplicates(subset=['tconst']).copy()

# For each movie it adds columns for the actor stats and crew stats.
movies_final = movies_unique.merge(actor_stats_df, on='tconst', how='left') \
    .merge(prod_stats_df, on='tconst', how='left')

# Removes the actor name ID, title name ID, person's name, and person's job category.
movies_final = movies_final.drop(columns=['nconst', "tconst", "primaryName", "category"])

print(movies_final.head())


       id            title  vote_average  vote_count    status release_date  \
0   27205        Inception          8.36       34495  Released    7/15/2010   
1  157336     Interstellar          8.42       32571  Released    11/5/2014   
2     155  The Dark Knight          8.51       30619  Released    7/16/2008   
3   19995           Avatar          7.57       29815  Released   12/15/2009   
4   24428     The Avengers          7.71       29166  Released    4/25/2012   

      revenue  runtime  adult     budget  ...  \
0   825532764      148  False  160000000  ...   
1   701729206      169  False  165000000  ...   
2  1004558444      152  False  185000000  ...   
3  2923706026      162  False  237000000  ...   
4  1518815515      143  False  220000000  ...   

                                production_companies  \
0  Legendary Pictures, Syncopy, Warner Bros. Pict...   
1  Legendary Pictures, Syncopy, Lynda Obst Produc...   
2  DC Comics, Legendary Pictures, Syncopy, Isobel...   
3  Dun

In [13]:
# Saves the final dataframe into a csv file.
movies_final.to_csv("movie-data/movies_with_scores.csv", index=False)
print("Saved movies_with_scores.csv")

Saved movies_with_scores.csv


In [14]:
print("Top 10 movies with highest actor average score:")
top_actor_avg = movies_final.sort_values('actor_avg', ascending=False).head(10)
print(top_actor_avg[['title', 'actor_avg', 'actor_med', 'actor_dev']])
print("Top 10 movies with highest production average score:")  
top_prod_avg = movies_final.sort_values('production_avg', ascending=False).head(10)
print(top_prod_avg[['title', 'production_avg', 'production_med', 'production_dev']])

Top 10 movies with highest actor average score:
                                                 title  actor_avg  actor_med  \
131                                      Despicable Me    1599.34    1013.79   
27                                       Black Panther    1433.77    1318.47   
550                                    Despicable Me 3    1418.72     800.60   
6                               Avengers: Infinity War    1390.47     887.31   
71                   The Hobbit: An Unexpected Journey    1285.68     929.93   
19   The Lord of the Rings: The Fellowship of the Ring    1211.49     782.12   
15                                   Avengers: Endgame    1209.33     887.31   
65        Harry Potter and the Deathly Hallows: Part 1    1195.00     775.56   
129                           Star Wars: The Last Jedi    1190.53     911.73   
179                The Hobbit: The Desolation of Smaug    1072.51     661.79   

     actor_dev  
131    1250.25  
27     1142.28  
550    1232.29  
6  